In [ ]:
# tulis ulang manual kode
# tanpa trigger reposisi RoI box
import cv2
from ultralytics import YOLO
global ROI_center_spot #mouse callback
import time
import psutil
import torch
import os



model = YOLO('runs/detect/yolov8s_vehicle_detector/weights/best.pt')
videopath = 'DJI_0004_HS0600D.MP4'

box_width = 640 #bounding box
box_height = 225

# GLOBAL VARIABLES
# core variables
prev_x = {} #ids in previous frame
counted_id = [] #id thats been counted after crossing the line
counted_vec_types = [] #types of vehicles of counted_id
counted_coordinates = [] #coordinate of vehicles of counted_id
counted_frame = [] #frame of vehicles of counted_id

# frame counting variables
current_frame = 0 # which frame is this
frame_fps = [] # how many second a frame is
frame_bounding_box_motor = [] #how many motorcycle bounding box in a frame
frame_bounding_box_car = [] #how many car bounding box in a frame

# frame time variables
frame_preprocess = []
frame_interference = []
frame_postprocess = []

# misc variables
false_positives_id = []

# performance variables
cpu_per_30 = []
ram_per_30 = []
vram_allocated_per_30 = []
process = psutil.Process(os.getpid())
process.cpu_percent #inisialisasi perhitungan CPU

# video variables
cap = cv2.VideoCapture(videopath)
if not cap.isOpened():
    print("Error: Could not open video stream.")
    exit()
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# RoI box coordinates configuration
ROI_center_spot = (744, video_height // 2) #center spot RoI box, posisi terbaik dengan pertimbangan pohon
ROI_top_left = (
    ROI_center_spot[0] - 30, 
    ROI_center_spot[1] - box_height // 2
)
ROI_bottom_right = (
    ROI_center_spot[0] + 50, #
    ROI_center_spot[1] + box_height // 2
)
ROI_x1, ROI_y1 = ROI_top_left
ROI_x2, ROI_y2 = ROI_bottom_right

# konfigurasi cv2 window (diluar loop)
cv2.namedWindow("YOLO Detection")

# loop next frame
while True:
    ret, frame = cap.read()

    if not ret:
        print('ERROR:cannot read frame')
        break

    frame_start = time.perf_counter()
    temp_counted_id = []
    motorcycle_count = 0
    car_count = 0
    
    # kode tracking
    roi = frame[ROI_y1:ROI_y2, ROI_x1:ROI_x2] # definisi RoI
    results = model.track(roi,
                          persist=True,
                          device=0, #menggunakan GPU
                          conf=0.3, #0.2~0.5
                          iou=0.5, #only iou 0.5 has been tested in result
                          max_det = 300, #
                          tracker='botsort.yaml')

    result = results[0]
    preprocess_ms = result.speed["preprocess"]
    inference_ms = result.speed["inference"]
    postprocess_ms = result.speed["postprocess"]

    frame_preprocess.append(preprocess_ms)
    frame_interference.append(inference_ms)
    frame_postprocess.append(postprocess_ms)

    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            if box.cls is None:
                continue
            
            if box.id is not None:
                track_id = int(box.id[0])
                cls = int(box.cls[0])
                conf = box.conf[0]
                label = f'{model.names[cls]} {conf:.2f}'

                if cls == 0:
                    motorcycle_count += 1
                elif cls == 1:
                    car_count += 1
            else:
                label = ""
            
            # perkiraan posisi bounding box
            bounding_x1, bounding_y1, bounding_x2, bounding_y2 = map(int, box.xyxy[0])

            # penyesuaian dengan frame awal
            bounding_x1 += ROI_x1
            bounding_y1 += ROI_y1
            bounding_x2 += ROI_x1
            bounding_y2 += ROI_y1

            center_bounding_x = (bounding_x1 + bounding_x2)//2
            center_bounding_y = (bounding_y1 + bounding_y2)//2

            # visualisasi bounding box pada cv2 window
            cv2.rectangle(frame, (bounding_x1, bounding_y1), (bounding_x2, bounding_y2), (0,0,255),2)
            cv2.putText(frame, label, (bounding_x1, bounding_y1 - 5),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),1)

            # counting line center spot
            counting_line_center = ROI_center_spot[0]

            # proses counting
            if track_id in prev_x:
                if center_bounding_x < counting_line_center:
                    if track_id not in counted_id:
                        counted_id.append(track_id)
                        counted_vec_types.append(label)
                        counted_frame.append(current_frame/fps)
                        counted_coordinates.append((center_bounding_x, center_bounding_y))
            elif track_id not in prev_x and track_id not in counted_id:
                false_positives_id.append(track_id)
            prev_x[track_id] = center_bounding_x
            

    display_counted_ids = list(counted_id)[-3:] if len(counted_id) > 3 else list(counted_id)        
    cv2.putText(frame, f"Counted IDs: {display_counted_ids}",
                        (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (255,0,0),
                        2)
    
    cv2.putText(frame, f"Total Counted Types: {len(counted_id)}",
                        (10, 110),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (255,0,0),
                        2)

    frame_bounding_box_motor.append(motorcycle_count)
    frame_bounding_box_car.append(car_count)
    
    # visualisasi bounding box dan counting line pada frame
    cv2.rectangle(frame, ROI_top_left, ROI_bottom_right,(0, 255, 0), 2) #visual bounding box
    cv2.line(frame, [ROI_center_spot[0], ROI_y1], [ROI_center_spot[0], ROI_y2], (0, 255, 0), 2) #visual counting line

    cv2.imshow('YOLO Detection', frame)

    # perhitungan performa setelah blok kode berjalan
    if current_frame % 30:
        cpu_percent = process.cpu_percent()
        ram_mb = process.memory_info().rss / (1024 ** 2)
        cpu_per_30.append(cpu_percent)
        ram_per_30.append(ram_mb)

        if torch.cuda.is_available():
            vram_allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
            vram_allocated_per_30.append(vram_allocated_mb)
        else:
            print('vram not available')


    frame_time_ms = (time.perf_counter() - frame_start)
    fps_per_frame = 1/frame_time_ms
    frame_fps.append(fps_per_frame)

    current_frame += 1
    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()    



0: 256x96 2 motorcycles, 2 cars, 135.7ms
Speed: 2.4ms preprocess, 135.7ms inference, 17.1ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 21.0ms
Speed: 1.0ms preprocess, 21.0ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 12.1ms
Speed: 1.3ms preprocess, 12.1ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 14.3ms
Speed: 1.8ms preprocess, 14.3ms inference, 1.8ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 11.3ms
Speed: 0.9ms preprocess, 11.3ms inference, 1.7ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 12.5ms
Speed: 0.9ms preprocess, 12.5ms inference, 1.6ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 13.3ms
Speed: 0.7ms preprocess, 13.3ms inference, 1.7ms postprocess per image at shape (1, 3, 256, 96)

0: 256x96 2 motorcycles, 2 cars, 19.3

In [13]:
print(next(model.model.parameters()).device)

cuda:0


In [4]:
import pandas as pd
df3 = pd.DataFrame({
    'motor': frame_bounding_box_motor,
    'mobil': frame_bounding_box_car
})
df3.to_excel('bounding per frame.xlsx')

In [5]:
import pandas as pd
df = pd.DataFrame({
    'Object ID' : counted_id,
    'Vehicle Types' : counted_vec_types,
    'Coordinates' : counted_coordinates,
    'Time' : counted_frame,
})

In [7]:
df.to_excel('hasildeteksi&id.xlsx')

In [6]:
dframe = pd.DataFrame({'fps per frame':frame_fps})
dframe.to_excel('fps per frame.xlsx')

In [10]:
import pandas as pd
df_time = pd.DataFrame({
    'preprocess time' : frame_preprocess,
    'interference time' : frame_interference,
    'postprocess time' : frame_postprocess,
})
df_time.to_excel('frame time.xlsx')

In [12]:
import pandas as pd
df_performance = pd.DataFrame({
    'CPU' : cpu_per_30,
    'RAM' : ram_per_30,
    'VRAM' : vram_allocated_per_30,
})
df_performance.to_excel('performance.xlsx')

In [11]:
print(false_positives_id)

[1, 2, 3, 4, 6, 8, 11, 12, 14, 15, 17, 18, 19, 20, 21, 23, 25, 27, 28, 29, 32, 33, 35, 37, 40, 45, 46, 48, 49, 50, 51, 52, 54, 56, 58, 59, 60, 63, 65, 67, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 83, 84, 85, 86, 87, 88, 89, 90, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 106, 107, 109, 111, 113, 116, 118, 119, 120, 123, 125, 127, 128, 130, 131, 132, 133, 134, 136, 138, 139, 140, 142, 144, 145, 149, 150, 151, 152, 155, 157, 158, 160, 162, 164, 165, 166, 167, 168, 169, 170, 171, 173, 175, 177, 178, 179, 180, 181, 182, 185, 186, 188, 191, 192, 194, 196, 197, 199, 202, 204, 206, 208, 209, 211, 213, 216, 217, 218, 219, 220, 221, 223, 225, 226, 227, 228, 230, 231, 232, 235, 236, 237, 238, 239, 240, 243, 244, 245, 246, 247, 249, 251, 252, 253, 254, 255, 258, 259, 261, 262, 265, 266, 269, 270, 271, 272, 273, 278, 283, 285, 289, 291, 292, 296, 298, 300, 301, 302, 304, 305, 306, 307, 310, 311, 312, 314, 315, 316, 317, 325, 329, 330, 333, 337, 338, 339, 340, 341, 342, 345, 347, 348, 350, 351, 352, 

<H4>Penghitungan Statistik


FPS


In [14]:
# P-95
import pandas as pd
import numpy as np

# fps_stat = pd.read_excel('important results\statistik bounding box (for py).xlsx')
fps_stat = pd.read_excel('fps per frame.xlsx')
fps_values = fps_stat['fps per frame']
p95_fps = np.percentile(fps_values, 95)
print(f"P95 FPS: {p95_fps:.2f}")

# average
avg_fps = np.average(fps_values)
print(f"Average FPS: {avg_fps:.2f}")


P95 FPS: 46.29
Average FPS: 34.92


Time per Frame


In [16]:
# Average
time_per_frame = pd.read_excel('frame time.xlsx')
avg_preprocess_time =np.average(time_per_frame['preprocess time'])
avg_interference_time = np.average(time_per_frame['interference time'])
avg_postprocess_time = np.average(time_per_frame['postprocess time'])
print(f"Average preprocess time: {avg_preprocess_time:.2f}")
print(f"Average interference time: {avg_interference_time:.2f}")
print(f"Average postprocess time: {avg_postprocess_time:.2f}")


# p95
p95_interference_time = np.percentile(time_per_frame['interference time'], 95)
print(f"P95 interference: {p95_interference_time:.2f}")


Average preprocess time: 1.57
Average interference time: 16.70
Average postprocess time: 2.20
P95 interference: 26.72


Performance

In [21]:
performance = pd.read_excel('performance.xlsx')

# Average
avg_cpu = np.average([performance['CPU']])
avg_ram = np.average(performance['RAM'])
avg_vram = np.average(performance['VRAM'])

print(f"Average CPU: {avg_cpu:.2f}%")
print(f"Average RAM: {avg_ram:.2f}MB")
print(f"Average VRAM: {avg_vram:.2f}MB")


Average CPU: 137.35%
Average RAM: 1529.90MB
Average VRAM: 42.46MB
